# Classifier Chain (Logistic Regression)
**text › multilabel** · notebook `classifier_chain_logreg`

**Kapan dipakai:** Memodelkan ketergantungan antar-label: prediksi label sebelumnya jadi fitur untuk label berikutnya.

**Jebakan umum:** Urutan rantai memengaruhi hasil; kesalahan merambat. Bandingkan dengan OvR sebelum dipercaya.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/text/multilabel/casa_train.csv"
TEST_PATH  = ROOT / "data_latihan/text/multilabel/casa_test.csv"
TEXT_COL   = "text"
TARGETS    = ['fuel', 'machine', 'others', 'part', 'price', 'service']         # kolom label biner (0/1), satu kolom per label
ID_COL     = 'id'
LANG       = "id"
USE_STOPWORDS = False
SEED, N_SPLITS, VALID_SIZE = 42, 5, 0.2
MODEL_KEY  = "classifier_chain_logreg"
OUT = out_dir("text", "multilabel", MODEL_KEY)
set_seed(SEED)
from superdb.text import ID_STOPWORDS, clean_text
STOP = (ID_STOPWORDS if LANG == "id" else "english") if USE_STOPWORDS else None

## 2. Baca data
Multilabel: satu teks boleh punya 0, 1, atau banyak label sekaligus.

In [ ]:
df = pd.read_csv(TRAIN_PATH); test = pd.read_csv(TEST_PATH) if TEST_PATH else None
Y = df[TARGETS].to_numpy(); X = df[TEXT_COL].map(clean_text)
print(df.shape, "| rata-rata label per teks:", round(Y.sum(1).mean(), 2), "| teks tanpa label:", int((Y.sum(1) == 0).sum()))
display(df[TARGETS].mean().round(3).to_frame("proporsi positif").T)

## 3. Pisahkan, lalu model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
Xtr, Xva, Ytr, Yva = train_test_split(X, Y, test_size=VALID_SIZE, random_state=SEED)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.multioutput import ClassifierChain
from sklearn.linear_model import LogisticRegression

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, strip_accents="unicode", stop_words=STOP)
model = ClassifierChain(LogisticRegression(C=5, max_iter=2000, class_weight='balanced'), order='random', random_state=42)
pipe = Pipeline([("vec", vec), ("model", model)])
pipe

## 4. Cross-validation

In [ ]:
from sklearn.model_selection import cross_validate, KFold
cvr = cross_validate(pipe, Xtr, Ytr, cv=KFold(N_SPLITS, shuffle=True, random_state=SEED), scoring=["f1_micro", "f1_macro"])
print({k: round(float(v.mean()), 4) for k, v in cvr.items() if k.startswith("test_")})

## 5. Evaluasi pada validation
Metrik: **f1_micro** (dominan oleh label sering), **f1_macro** (setiap label setara), **hamming loss** (porsi sel label yang salah), **subset accuracy** (semua label benar sekaligus; keras).

In [ ]:
from sklearn.metrics import f1_score, hamming_loss, accuracy_score
pipe.fit(Xtr, Ytr); P = pipe.predict(Xva)
metrics = {"f1_micro": f1_score(Yva, P, average="micro", zero_division=0), "f1_macro": f1_score(Yva, P, average="macro", zero_division=0),
           "hamming_loss": hamming_loss(Yva, P), "subset_accuracy": accuracy_score(Yva, P)}
display(show_table(metrics))
display(pd.DataFrame({"f1": f1_score(Yva, P, average=None, zero_division=0), "support": Yva.sum(0)}, index=TARGETS).round(3))

## 6. Threshold per label
Untuk label langka, ambang bawaan sering salah. Ambang dipilih dari prediksi **out-of-fold pada training**, lalu dipakai pada validation.

In [ ]:
has_p = hasattr(pipe, "predict_proba")
score = (lambda m, X_: m.predict_proba(X_)) if has_p else (lambda m, X_: m.decision_function(X_))
from sklearn.base import clone
oof = np.zeros(Ytr.shape, dtype=float)                       # OOF manual (cross_val_predict tidak kompatibel dengan ClassifierChain)
for tr_i, va_i in KFold(N_SPLITS, shuffle=True, random_state=SEED).split(Xtr):
    oof[va_i] = score(clone(pipe).fit(Xtr.iloc[tr_i], Ytr[tr_i]), Xtr.iloc[va_i])
THR = []
for j in range(len(TARGETS)):
    cand = np.quantile(oof[:, j], np.linspace(0.05, 0.95, 60))
    THR.append(cand[int(np.argmax([f1_score(Ytr[:, j], oof[:, j] >= t, zero_division=0) for t in cand]))])
S = score(pipe, Xva); P2 = (S >= np.array(THR)).astype(int)
print(f"f1_micro: bawaan {metrics['f1_micro']:.4f} -> threshold per label {f1_score(Yva, P2, average='micro', zero_division=0):.4f} | "
      f"f1_macro: {metrics['f1_macro']:.4f} -> {f1_score(Yva, P2, average='macro', zero_division=0):.4f}")
metrics["f1_micro_thr"] = f1_score(Yva, P2, average="micro", zero_division=0); metrics["f1_macro_thr"] = f1_score(Yva, P2, average="macro", zero_division=0)

## 7. Latih ulang pada semua data → submission → simpan

In [ ]:
from sklearn.base import clone
final = clone(pipe).fit(X, Y)
if test is not None:
    Pt = (score(final, test[TEXT_COL].map(clean_text)) >= np.array(THR)).astype(int)
    sub = pd.DataFrame(Pt, columns=TARGETS); sub.insert(0, ID_COL, test[ID_COL].to_numpy())
    sub.to_csv(OUT / "submission.csv", index=False); display(sub.head(3))
save_model({"pipe": final, "thresholds": THR, "targets": TARGETS}, OUT / "model.joblib")
save_json({"valid": metrics, "thresholds": dict(zip(TARGETS, map(float, THR)))}, OUT / "metrics.json")
b = load_model(OUT / "model.joblib"); assert (b["pipe"].predict(X.head(30)) == final.predict(X.head(30))).all(); print("OK. hasil di", OUT)

## Langkah berikutnya
- Bandingkan dengan model lain di folder ini; pilih berdasarkan CV.
- Format submission lomba bisa berupa daftar label dipisah spasi: ubah dari matriks 0/1 di sel terakhir.